In [0]:
%pip install xgboost scikit-optimize shap --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# COMMAND ----------
# 00 - SETUP & DATA LOAD

import pandas as pd
import numpy as np
import gc
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier
from skopt import BayesSearchCV
from skopt.space import Real, Integer
import shap
import mlflow
import mlflow.sklearn

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
print(f"✅ Spark session initialized (version {spark.version})")

# COMMAND ----------
# Load final AP dataset (your working dataset)

spark_df = (
    spark.read.format("csv")
    .option("header", "true")
    .load("/Volumes/fraud_project/default/fraud_data/final_ap_dataset.csv")
)

df = spark_df.toPandas()
print(f"✅ Loaded {df.shape[0]:,} rows with {df.shape[1]} columns")

del spark_df
gc.collect()

# COMMAND ----------
# Core cleaning + base feature engineering (from your code)

df["date"] = pd.to_datetime(df["date"], format="mixed", dayfirst=False)

numeric_cols = ['amount', 'duplicate_flag', 'invoice_count', 'avg_amount', 'weekend_flag']
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month

df["high_amount_flag"] = (df["amount"] > (df["avg_amount"] * 3)).astype(int)
df["is_weekend"] = df["date"].dt.dayofweek.isin([5, 6]).astype(int)

df["bank_change_flag"] = np.random.choice([0, 1], len(df), p=[0.95, 0.05])
df["ghost_vendor_flag"] = np.random.choice([0, 1], len(df), p=[0.98, 0.02])

vendor_counts = df.groupby("vendor_id")["invoice_id"].transform("count")
df["velocity_spike"] = (vendor_counts > vendor_counts.mean() * 3).astype(int)

df["first_digit"] = df["amount"].astype(str).str[0].astype(int)
benford_dist = {1:0.301,2:0.176,3:0.125,4:0.097,5:0.079,6:0.067,7:0.058,8:0.051,9:0.046}
df["benford_score"] = df["first_digit"].map(benford_dist)

# Persist engineered frame via parquet file to avoid driver OOM on spark.createDataFrame()
# Strip Spark Connect metadata and coerce timestamps for Spark compatibility
df_clean = pd.DataFrame({col: df[col].values for col in df.columns})
tmp_path = "/Volumes/fraud_project/default/fraud_data/_tmp_engineered_base.parquet"
table = pa.Table.from_pandas(df_clean, preserve_index=False)
pq.write_table(table, tmp_path, coerce_timestamps="us", allow_truncated_timestamps=True)
print(f"✅ Wrote {len(df_clean):,} rows to temp parquet")

del df, df_clean, table
gc.collect()

spark.sql(f"""
    CREATE OR REPLACE TABLE fraud_project.default.engineered_base
    AS SELECT * FROM read_files('{tmp_path}', format => 'parquet')
""") 
print("✅ Saved fraud_project.default.engineered_base table")

✅ Spark session initialized (version 4.0.0)
✅ Loaded 7,173,725 rows with 9 columns
✅ Wrote 7,173,725 rows to temp parquet


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5247704600956809>, line 77
     74 del df, df_clean, table
     75 gc.collect()
---> 77 spark.read.parquet(tmp_path).write.mode("overwrite").saveAsTable("fraud_project.default.engineered_base")
     78 print("✅ Saved fraud_project.default.engineered_base table")

File /databricks/spark/python/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/spark/python/pyspark/sql/connect/client/core.py:1589, in SparkConnectClient.execute_command(self, command, observations, extra_request_met